# Neural Crest

## Purpose

Determine how different neural crest cell types are specified during development

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycea as py
import scanpy as sc
import seaborn as sns
from devmap.config import (
    colors,
    germ_layer_palette,
    get_paths,
    lineage_palette,
    sequential_cmap,
    set_theme,
    stage_palette,
    subtype_palette,
)
from devmap.fate import annotate_commitment, extant_transition_count
from devmap.plots import plot_bipotent_probability_over_time, plot_progenitor_output, plot_restriction_sankey
from devmap.progenitors import bipotency_permutation_test, get_fate_progenitors
from devmap.utils import load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("vignettes")

## Load data

In [3]:
tdata = load_data("topology",scvi = True)

## Annotate NC type

In [ ]:
nc_lineage_map = {
    # Neural crest (undifferentiated / axial identity)
    "Cranial neural crest": "Axial",
    "Vagal neural crest": "Axial",
    "Trunk neural crest": "Axial",
    "Sacral neural crest": "Axial",
    # Mesenchymal (ectomesenchyme)
    "Frontonasal mesenchyme": "Mesenchymal",
    "Pharyngeal arch mesenchyme": "Mesenchymal",
    # Sensory lineage
    "Dorsal root ganglion progenitor": "Sensory",
    "Dorsal root ganglion neuron": "Sensory",
    "Trigeminal ganglion neuron": "Sensory",
    # Autonomic lineage
    "Autonomic progenitor": "Autonomic",
    "Autonomic neuron": "Autonomic",
}
nc_axial_map = {
    # Neural crest (explicit axial identity)
    "Cranial neural crest": "Cranial",
    "Vagal neural crest": "Vagal",
    "Trunk neural crest": "Trunk",
    "Sacral neural crest": "Sacral",
    # Mesenchymal derivatives (cranial NC-derived)
    "Frontonasal mesenchyme": "Cranial",
    "Pharyngeal arch mesenchyme": "Cranial",
    # Sensory lineage
    "Trigeminal ganglion neuron": "Cranial",
    "Dorsal root ganglion progenitor": "Trunk",
    "Dorsal root ganglion neuron": "Trunk",
    # Autonomic lineage (mostly trunk/sacral; vagal contributes enteric but not listed here)
    "Autonomic progenitor": pd.NA,
    "Autonomic neuron": pd.NA,
}
tdata.obs["is_nc"] = (tdata.obs["lineage"] == "Neural crest")
tdata.obs["nc_lineage"] = tdata.obs["cell_type"].map(nc_lineage_map)
tdata.obs["nc_axial"] = tdata.obs["cell_type"].map(nc_axial_map)
tdata.obs["nc_type"] = tdata.obs["nc_axial"].replace({"Vagal":"Trunk", "Sacral":"Trunk", pd.NA:"Trunk"})
trunk_tdata = tdata[tdata.obs["nc_axial"].isin(["trunk","vagal","sacral"]) | (tdata.obs["nc_lineage"] == "autonomic"),].copy()
nc_palette = {"Autonomic":  '#ff8a9a',"Sensory":  "#00FDF5", "Autonomic-biased":  "#ff00ff","Sensory-biased":  "#66CCCC",
              "Bipotent":  "#FFAA00","Mesenchymal": "#a30059", "Mesenchymal-biased": "#94617d", "Independent": "black",
              "DRG neuron/prog.": "#00FDF5", "Autonomic neuron/prog.": "#ff8a9a"}

Plot number of cells per stage and type

In [ ]:
fig, axes = plt.subplots(3,1, figsize=(1,1.5), dpi=600, sharex=True, sharey=True)
for i, pop in enumerate(["vagal", "trunk", "sacral"]):
    use_stages = ["E8.5", "E9.0", "E9.5"]
    counts = (
        trunk_tdata.obs.query("stage in @use_stages")
        .query(f"prog_type == '{pop}'")
        .groupby(["stage", "nc_lineage"])
        .size()
        .unstack(fill_value=0)
    )
    # 2. Plot stacked bar
    counts.loc[["E9.5","E9.0", "E8.5"], ["axial","autonomic", "sensory",]].plot(kind="barh", stacked=True,
        color = ["dimgray", '#ff8a9a', colors[1]], ax=axes[i], legend=False, width=0.8)
    axes[i].set_ylabel(pop)
    axes[i].set_xticks([0,5000,10000])
save_plot(plots_path / "nc_prog_stacked_bar.svg", fig, rasterize=False)

## Identify fate-restricted clades

In [ ]:
tdata.obs["is_nc"] = tdata.obs["is_nc"].astype(int)
py.tl.ancestral_states(tdata, "is_nc", method = "mean")
tdata.obs["is_nc"] = tdata.obs["is_nc"].astype(bool)
nc_progenitors = get_fate_progenitors(tdata, "is_nc", key_added = "nc_progenitor")
nc_progenitors["stage"] = nc_progenitors["clone"].str.split("-R").str[0]
nc_progenitors["embryo"] = nc_progenitors["clone"].str.split("-C").str[0]
tdata.obs["progenitor_axial"] = tdata.obs.query("is_nc")["nc_progenitor"].map(
    tdata.obs.query("nc_axial.notnull()").groupby("nc_progenitor")["nc_axial"].agg(lambda x: x.mode()[0]))
nc_progenitors["axial"] = nc_progenitors.node.map(
    tdata.obs.query("nc_axial.notnull()").groupby("nc_progenitor")["nc_axial"].agg(lambda x: x.mode()[0]))
nc_progenitors["nc_type"] = nc_progenitors["axial"].replace({"Vagal":"Trunk", "Sacral":"Trunk"})
trunk_progenitors = nc_progenitors.query("stage == 'E9.5' & node in @trunk_nodes")

tree with clades

In [ ]:
fig, ax = plt.subplots(figsize=(1.2, 3.6), dpi = 600)
clone = 'E9.5-R2-C1'
nodes = tdata.obs.query("clone == @clone & nc_progenitor.notnull()").nc_progenitor.unique().tolist()
py.pl.tree(tdata, depth_key="time", tree = clone, ax = ax, branch_linewidth=.5)
py.pl.nodes(tdata,tree = clone, nodes = nodes, ax = ax, color = lineage_palette["Neural crest"], size = 15, outline_width=.3)
save_plot(plots_path / "nc_progenitor_tree.svg", fig, rasterize=True)

Example clades

In [ ]:
for node in ["6zDg6AcR","Q08xb6iQ","unxYrEJj"]:
    subset = tdata[tdata.obs.progenitor == node].copy()
    fig, ax = plt.subplots(figsize=(1.5,subset.shape[0]/10), dpi=600)
    py.pl.tree(subset, depth_key="time", ax = ax, branch_linewidth=0.5)
    py.pl.annotation(subset, keys = "cell_subtype", palette = subtype_palette, label = False, ax = ax, gap = .05, legend = True, width = .15)
    save_plot(plots_path / f"nc_prog_tree_{node}.svg", fig)

## Fate restriction time

In [ ]:
fig, axes = plt.subplots(3,1, figsize=(1.5,1.5), dpi=600, sharex=True, sharey=False)
for i, pop in enumerate(["vagal", "trunk", "sacral"]):
    sns.kdeplot(data=nc_progenitors.query(f"prog_type == '{pop}'"), x="time", hue="stage",
                palette=stage_palette, ax=axes[i], legend=False)
    plt.xlim(6, 10)
    axes[i].set_ylabel(pop)
    # add dashed line with mean 
    mean_time = nc_progenitors.query(f"prog_type == '{pop}'")["time"].mean()
    axes[i].axvline(mean_time, color="black", linestyle="--", linewidth=0.8)
    # add text with mean time
    plt.text(mean_time + 0.1, 0.5, f"{mean_time:.2f}", transform=axes[i].get_xaxis_transform(), color="black")
save_plot(plots_path / "nc_prog_kde.svg", fig, rasterize=False)

## Fate distribution heatmap

In [ ]:
def neural_crest_group(frac):
    groups = np.select(
        [
            (frac["axial"] > 0) & (frac["sensory"] == 0) & (frac["autonomic"] == 0),
            (frac["sensory"] > 0) & (frac["autonomic"] == 0),
            (frac["autonomic"] > 0) & (frac["sensory"] == 0),
            (frac["sensory"] > 0) & (frac["autonomic"] > 0),
        ],
        ["axial-only", "axial-sensory", "axial-autonomic", "all three"],
        default="other",
    )
    return pd.Categorical(
        groups,
        categories=["axial-only", "axial-sensory", "axial-autonomic", "all three", "other"],
        ordered=True,
    )

df = trunk_tdata.obs.query("stage == 'E9.5'").copy()

counts = (
    df.groupby(["progenitor", "nc_lineage"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["axial", "sensory", "autonomic"], fill_value=0)
)

counts = counts[counts.sum(axis=1) >= 5]
frac = counts.div(counts.sum(axis=1), axis=0)

keep = frac.index[neural_crest_group(frac) != "other"]
df = df[df["progenitor"].isin(keep)]

fig = plot_progenitor_output(
    df,
    progenitor_col="progenitor",
    fate_col="nc_lineage",
    fate_order=["axial", "sensory", "autonomic"],
    min_cells=5,
    row_order="sort",
    row_group_fn=neural_crest_group,
    sort_col="axial",
    cmap=sequential_cmap,
    ylim=(0, 80),
    yticks=[0, 40, 80],
    xlabel="Posterior neural crest progenitors",
)

save_plot(plots_path / "trunk_nc_cooccurrence_heatmap.svg", fig, rasterize=True)

## UMAP

In [ ]:
sc.pp.neighbors(trunk_tdata, use_rep="X_scvi")
sc.tl.umap(trunk_tdata)

Colored by stage

In [ ]:
fig, ax = plt.subplots(figsize=(4,4), dpi = 600)
sc.pl.umap(trunk_tdata, color=["stage"], palette = stage_palette, 
    sort_order = False, ax = ax, title = "", frameon=False)
save_plot(plots_path / "trunk_nc_stage_umap.svg", fig, rasterize=True)

Colored by subtype

In [ ]:
fig, ax = plt.subplots(figsize=(4,4), dpi = 600)
replacements = {"Dorsal root ganglion neuron": "DRG neuron/prog.",
                "Dorsal root ganglion progenitor": "DRG neuron/prog.",
                "Autonomic progenitor": "Autonomic neuron/prog.",
                "Autonomic neuron": "Autonomic neuron/prog."}
trunk_tdata.obs["cell_subtype"] = trunk_tdata.obs["cell_type"].replace(replacements)
sc.pl.umap(trunk_tdata, color=["cell_subtype"], palette = subtype_palette, 
    sort_order = False, ax = ax, title = "", frameon=False)
save_plot(plots_path / "trunk_nc_subtype_umap.svg", fig, rasterize=True)

Example bipotent clades

In [ ]:
fig, ax = plt.subplots(figsize=(4,4), dpi=600)
example_progs = frac.query("group == 'all three'").sample(10).index.to_list()
sc.pl.umap(trunk_tdata, ax = ax, title = "", frameon=False, show = False)
sc.pl.umap(trunk_tdata[trunk_tdata.obs.progenitor.isin(example_progs)], s = 100,
    color = "progenitor", groups = example_progs, ax = ax, title = "", frameon=False, palette = colors)
save_plot(plots_path / "nc_bipotent_umap.svg", fig, rasterize=True)

Example DRG-biased clades

In [ ]:
fig, ax = plt.subplots(figsize=(4,4), dpi=600)
example_progs = frac.query("group == 'axial-sensory'").sample(10).index.to_list()
sc.pl.umap(trunk_tdata, ax = ax, title = "", frameon=False, show = False)
sc.pl.umap(trunk_tdata[trunk_tdata.obs.progenitor.isin(example_progs)], s = 100,
    color = "progenitor", groups = example_progs, ax = ax, title = "", frameon=False, palette = colors)
save_plot(plots_path / "nc_drg_umap.svg", fig, rasterize=True)

## Test NC potential

Biopotent fraction barplot

In [ ]:
fig, ax = plt.subplots(figsize=(1, 1.5), dpi=600)
res = bipotency_permutation_test(
    trunk_progenitors.query("stage == 'E9.5'"),
    lineages=("Autonomic", "Sensory"),
    ax=ax
)
save_plot(plots_path / "trunk_bipotent_barplot.svg", fig)

Bipotency probability over time

In [ ]:
fig, ax = plt.subplots(figsize=(1.8, 1.8), dpi=600)
fig = plot_bipotent_probability_over_time(
    nc_progenitors.query("stage == 'E9.5' & nc_type == 'Trunk'"),
    lineages=("Autonomic", "Sensory"),
    palette = nc_palette, ax=ax)
save_plot(plots_path / "trunk_bipotent_probability_over_time.svg", fig)

## Sankey plot

In [526]:
e95_tdata = tdata[(tdata.obs.stage == "E9.5") & tdata.obs.clone.notnull(),[]].copy()
germ_assignment = e95_tdata.obs["germ_layer"].map(lambda x: "Ectoderm" if x == "Ectoderm" else "Other").to_dict()
lineage_bipotent = e95_tdata.obs["lineage"].map(lambda x: "Bipotent" if x == "Neural crest" else "Other")
lineage_biased = (e95_tdata.obs["nc_lineage"].fillna("Other").replace({"Axial":pd.NA,"Mesenchymal":pd.NA}) + "-biased").to_dict()
lineage_assignment = e95_tdata.obs["nc_lineage"].fillna("Other").replace({"Axial":"Other","Mesenchymal":"Other"}).to_dict()
for tree in e95_tdata.obst.values():
    annotate_commitment(tree, [germ_assignment, lineage_bipotent, lineage_biased, lineage_assignment], threshold=0.9, exclude_categories=["Other","Other-biased"])

In [543]:
paths = extant_transition_count(e95_tdata[e95_tdata.obs.query("nc_lineage.isin(['Autonomic','Sensory']) & nc_type == 'Trunk'").index], key = "commitment",
                                depth_key="time", bins = np.arange(0, 10, .5))
fig, ax = plt.subplots(figsize=(2.8, 2),dpi=600)
sankey_palette = sankey_palette = {**germ_layer_palette,**lineage_palette,**nc_palette,**{"Uncommitted": "#CCCCCC"}}
sankey_order = ["Uncommitted", "Ectoderm", "Bipotent","Autonomic-biased", "Autonomic","Sensory-biased", "Sensory",]
plot_restriction_sankey(paths.query("source_time > 3 & target != 'Mesenchymal'"), normalize = True, ax = ax, order = sankey_order, palette = sankey_palette)
plt.xticks([1,5,9,13], ["4", "6", "8", "10"]);
plt.xlim(0, 13)
plt.xticks(rotation=0)
save_plot(plots_path / "trunk_differentiated_nc_sankey.svg", fig, rasterize = False)